# **Project Name**

## Flipkart Customer Support CSAT Analysis & Prediction


# **Project Summary -**


This project analyzes customer-support interactions to understand the factors associated with Customer Satisfaction (CSAT) and to build a machine learning model that identifies interactions likely to result in a satisfied customer. In a competitive e-commerce environment, faster responses, effective issue handling, capable support teams, and appropriate channel strategies are important for customer retention and loyalty.

The dataset contains 85,907 customer-support records and 20 variables covering support channel, issue category and sub-category, customer remarks, order information, timestamps, customer city, product category, item price, connected handling time, agent hierarchy, tenure, shift, and CSAT score. CSAT ranges from 1 to 5. For machine learning, the score is converted into a business-oriented binary target: scores 4-5 are classified as “Satisfied” and scores 1-3 as “Not Satisfied”. This creates a practical outcome for identifying interactions that may require service recovery.

# **GitHub Link -**


GitHub link was not provided.


# **Problem Statement**


**Problem Statement**

In a competitive e-commerce environment, customer satisfaction is a critical driver of loyalty and retention. The objective is to analyze customer-support interactions, identify the operational and categorical factors associated with CSAT, and build a classification model that predicts whether an interaction is likely to receive a satisfied CSAT score (4-5) or a not-satisfied score (1-3).

The project answers:
- Which support channels and issue categories are associated with higher or lower CSAT?
- How do response time, product category, agent tenure, and shift relate to satisfaction?
- Are the observed differences statistically significant?
- Can support-interaction attributes predict satisfaction reliably?
- Which factors should management prioritize for service improvement?


# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import os, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind, mannwhitneyu, kruskal
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.inspection import permutation_importance
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42


### Dataset Loading

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load Dataset
data_path= '/content/drive/MyDrive/Masters/Introduction to Machine Learning (1)/Customer_support_data.csv'
if data_path is None:
    raise FileNotFoundError("Customer_support_data.csv was not found. Place the CSV beside this notebook or upload it.")
df = pd.read_csv(data_path)
print(f"Loaded: {data_path}")
print(f"Shape: {df.shape}")


### Dataset First View

In [ ]:
df.head()


### Dataset Rows & Columns count

In [ ]:
print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]:,}')


### Dataset Information

In [ ]:
df.info()


#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate Unique id values:", df["Unique id"].duplicated().sum())


#### Missing Values/Null Values

In [ ]:
missing_summary = pd.DataFrame({"Missing_Count": df.isna().sum(), "Missing_%": (df.isna().mean()*100).round(2)})
missing_summary.sort_values("Missing_Count", ascending=False)


In [ ]:
plt.figure(figsize=(12,6))
missing_pct=(df.isna().mean()*100).sort_values(ascending=True)
plt.barh(missing_pct.index,missing_pct.values)
plt.title("Missing Values by Column"); plt.xlabel("Missing Values (%)"); plt.ylabel("Column")
plt.tight_layout(); plt.show()


### What did you know about your dataset?

The dataset has 85,907 rows and 20 columns. It contains three support channels, 12 issue categories, 57 sub-categories, customer feedback, order/timestamp fields, product and price information, agent hierarchy, tenure, shift, and a 1-5 CSAT score. The target is highly concentrated at score 5. Missingness is substantial in several operational fields, so data quality and appropriate imputation are central to the analysis.


## ***2. Understanding Your Variables***

In [ ]:
pd.DataFrame({'Column': df.columns, 'Data Type': df.dtypes.astype(str).values})


In [ ]:
df.describe(include='all').T


### Variables Description

The dataset contains 85,907 support interactions and 20 variables. It covers support channels, issue categories and sub-categories, customer remarks, order and timestamp fields, product information, agent hierarchy, tenure, shifts, and CSAT. CSAT has five levels from 1 to 5 and is strongly concentrated at score 5, so class imbalance is important for classification.

Several fields have substantial missingness. Customer Remarks, customer city, product category, item price, order timestamps, and connected handling time are incomplete. Connected handling time is especially sparse and is therefore treated as a data-availability indicator rather than a core numeric predictor.


### Check Unique Values for each variable.

In [ ]:
unique_summary = pd.DataFrame({
    "Column": df.columns,
    "Unique Values": [df[c].nunique(dropna=True) for c in df.columns],
    "Missing Values": [df[c].isna().sum() for c in df.columns]
}).sort_values("Unique Values", ascending=False)
unique_summary


## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Make the dataset analysis ready
data = df.copy()

# Parse timestamps using the formats present in the source data.
data["order_dt"] = pd.to_datetime(data["order_date_time"], format="%d/%m/%Y %H:%M", errors="coerce")
data["issue_dt"] = pd.to_datetime(data["Issue_reported at"], format="%d/%m/%Y %H:%M", errors="coerce")
data["response_dt"] = pd.to_datetime(data["issue_responded"], format="%d/%m/%Y %H:%M", errors="coerce")
data["survey_dt"] = pd.to_datetime(data["Survey_response_Date"], format="%d-%b-%y", errors="coerce")

# Calculate response time and invalidate impossible negative values.
data["response_mins"] = (data["response_dt"] - data["issue_dt"]).dt.total_seconds()/60
data.loc[data["response_mins"] < 0, "response_mins"] = np.nan

# Clean text/categorical fields.
data["Customer Remarks"] = data["Customer Remarks"].fillna("").astype(str).str.strip()
for col in ["channel_name","category","Sub-category","Customer_City","Product_category",
            "Agent_name","Supervisor","Manager","Tenure Bucket","Agent Shift"]:
    data[col] = data[col].fillna("Unknown").astype(str).str.strip()

# Robust numeric transformations.
response_cap = data["response_mins"].quantile(.99)
price_cap = data["Item_price"].quantile(.99)
data["response_mins_capped"] = data["response_mins"].clip(upper=response_cap)
data["item_price_capped"] = data["Item_price"].clip(lower=0, upper=price_cap)
data["item_price_log"] = np.log1p(data["item_price_capped"])

# Operational features.
data["remark_length"] = data["Customer Remarks"].str.len()
data["remark_word_count"] = data["Customer Remarks"].str.split().str.len()
data["has_customer_remark"] = (data["Customer Remarks"].str.len() > 0).astype(int)
data["has_order"] = data["Order_id"].notna().astype(int)
data["handling_time_available"] = data["connected_handling_time"].notna().astype(int)
data["issue_hour"] = data["issue_dt"].dt.hour
data["issue_dayofweek"] = data["issue_dt"].dt.dayofweek
data["is_weekend"] = (data["issue_dayofweek"] >= 5).astype(int)
data["issue_month"] = data["issue_dt"].dt.month

# Business target: CSAT 4-5 = satisfied; 1-3 = not satisfied.
data["Satisfied"] = (data["CSAT Score"] >= 4).astype(int)
data["Satisfaction_Label"] = data["Satisfied"].map({0:"Not Satisfied (1-3)",1:"Satisfied (4-5)"})

print("Prepared shape:", data.shape)
print("Invalid negative response times handled:", int((data["response_dt"] < data["issue_dt"]).sum()))
display(data[["CSAT Score","Satisfied","Satisfaction_Label"]].head())


### What all manipulations have you done and insights you found?

- Checked exact duplicates and duplicate unique identifiers.
- Parsed issue, response, order and survey timestamps.
- Treated negative response times as invalid and converted them to missing.
- Created response-time, hour, weekday/weekend, order-presence, remark-length and handling-time-availability features.
- Capped extreme response time and item price at the 99th percentile and log-transformed item price.
- Converted CSAT into a business-friendly binary target while retaining the original 1-5 score for EDA.
- Excluded identifier-like fields from predictive modeling to reduce leakage/noise.


## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
counts=data["CSAT Score"].value_counts().reindex([1,2,3,4,5],fill_value=0)
plt.figure(figsize=(8,5)); plt.bar(counts.index,counts.values)
plt.title("CSAT Score Distribution"); plt.xlabel("CSAT Score"); plt.ylabel("Interactions")
plt.tight_layout(); plt.show()


A count plot is appropriate for the discrete ordinal CSAT target and clearly reveals the distribution and class concentration.


A bar chart is a simple and readable comparison of average CSAT across support channels.


CSAT 5 is the dominant response, while scores 2 and 3 are much less frequent. The target is therefore strongly skewed toward high satisfaction.


Email records have the lowest average CSAT (about 3.90), while Inbound and Outcall are above 4.25. This is a clear channel-level service-quality signal.


The overall average is positive, but the CSAT 1 segment is large enough to represent a meaningful service-recovery opportunity. Low-score drivers should be investigated rather than relying only on the overall average.


Yes. The email channel is a practical improvement opportunity. The result suggests reviewing email response SLAs, resolution quality and customer communication, while avoiding causal claims from a descriptive chart alone.


#### Chart - 2

In [ ]:
channel_csat=data.groupby("channel_name")["CSAT Score"].mean().sort_values(ascending=False)
plt.figure(figsize=(8,5)); plt.bar(channel_csat.index,channel_csat.values)
plt.title("Average CSAT by Support Channel"); plt.xlabel("Channel"); plt.ylabel("Average CSAT"); plt.ylim(0,5)
plt.tight_layout(); plt.show()


A bar chart gives a direct comparison of average CSAT across the three support channels.


A horizontal bar chart keeps the 12 issue categories readable and supports direct comparison of average CSAT.


Email has the lowest average CSAT, while Inbound and Outcall are higher. The email journey should be investigated for response speed, resolution quality, and communication clarity.


Returns and Payments related issues have relatively high average CSAT, while Others and Cancellation are comparatively low.


Improving a lower-performing channel can raise satisfaction. The result is directional and should be validated after controlling for issue mix.


Category-level differences can guide specialized workflows and training. Low-performing categories should be investigated for recurring process or policy issues.


#### Chart - 3

In [ ]:
cat_csat=data.groupby("category")["CSAT Score"].mean().sort_values()
plt.figure(figsize=(10,6)); plt.barh(cat_csat.index,cat_csat.values)
plt.title("Average CSAT by Issue Category"); plt.xlabel("Average CSAT"); plt.ylabel("Issue Category"); plt.xlim(0,5)
plt.tight_layout(); plt.show()


The category bar chart makes it easy to compare satisfaction across major customer reasons for contacting support.


A bar chart is suitable for comparing the average CSAT of distinct tenure groups.


Returns and Payments related interactions have relatively strong CSAT, while Cancellation and Others are comparatively lower. Smaller categories should be interpreted cautiously.


Agents in On Job Training have a lower average CSAT than several more experienced groups, while 61-90 is among the strongest groups.


Category-level differences support targeted playbooks and training. Repeated low performance in a category can create negative customer experience and should receive root-cause analysis.


Targeted onboarding and coaching can improve early-stage performance. Tenure is associated with satisfaction here, but other factors such as issue mix may explain part of the difference.


#### Chart - 4

In [ ]:
tenure_order=["On Job Training","0-30","31-60","61-90",">90"]
vals=data.groupby("Tenure Bucket")["CSAT Score"].mean().reindex(tenure_order)
plt.figure(figsize=(8,5)); plt.bar(vals.index,vals.values)
plt.title("Average CSAT by Agent Tenure"); plt.xlabel("Tenure Bucket"); plt.ylabel("Average CSAT"); plt.ylim(0,5)
plt.tight_layout(); plt.show()


Tenure is actionable through onboarding and coaching, and a bar chart clearly compares satisfaction across tenure groups.


Shift is a categorical staffing variable, and a bar chart makes the average-CSAT differences easy to interpret.


On Job Training has lower average CSAT than several experienced tenure groups, while 61-90 is among the stronger groups. Experience may be associated with better service familiarity.


Split has the highest observed average CSAT, while Morning is comparatively lower. The differences are modest but operationally useful.


Improving onboarding and coaching can create positive impact, but tenure should not be interpreted causally without controlling for issue and channel mix.


Shift-level insights can support staffing and coaching decisions, but should be combined with workload and response-time analysis before changing schedules.


#### Chart - 5

In [ ]:
shift_csat=data.groupby("Agent Shift")["CSAT Score"].mean().sort_values(ascending=False)
plt.figure(figsize=(8,5)); plt.bar(shift_csat.index,shift_csat.values)
plt.title("Average CSAT by Agent Shift"); plt.xlabel("Agent Shift"); plt.ylabel("Average CSAT"); plt.ylim(0,5)
plt.tight_layout(); plt.show()


Shift is a staffing variable, so a bar chart is an intuitive way to compare average satisfaction across shifts.


Product categories represent different customer journeys and issue complexity, so comparing their average CSAT is useful.


The Split shift has the highest observed average CSAT and Morning is comparatively lower. The differences are useful for staffing analysis but are not necessarily causal.


Mobile, Furniture and Home Appliances have comparatively lower average CSAT. Very small categories should be treated cautiously.


Shift-level patterns can inform workforce planning and coaching, especially when combined with response-time and workload data.


Product-specific troubleshooting and service policies can improve weaker product journeys, but decisions should consider category volume and issue mix.


#### Chart - 6

In [ ]:
prod=data.groupby("Product_category")["CSAT Score"].mean().sort_values()
plt.figure(figsize=(9,5)); plt.barh(prod.index,prod.values)
plt.title("Average CSAT by Product Category"); plt.xlabel("Average CSAT"); plt.ylabel("Product Category"); plt.xlim(0,5)
plt.tight_layout(); plt.show()


Product category can affect issue complexity and customer expectations, making it useful for segmentation.


A box plot shows median and spread and is more informative than a mean alone when response time is skewed.


Mobile, Furniture and Home Appliances show relatively lower average CSAT than several other categories. Very small categories need cautious interpretation.


Satisfied interactions have a much lower typical response time than non-satisfied interactions. The long tail indicates that delayed responses are an important operational issue.


Product-specific troubleshooting and clearer service policies may improve low-performing product journeys, but decisions should consider category volume.


Reducing response delays is a direct service-quality opportunity. Response time should still be interpreted alongside issue complexity and resolution success.


#### Chart - 7

In [ ]:
plot_df=data.dropna(subset=["response_mins_capped"])
groups=[plot_df.loc[plot_df["Satisfied"]==0,"response_mins_capped"],plot_df.loc[plot_df["Satisfied"]==1,"response_mins_capped"]]
plt.figure(figsize=(8,5)); plt.boxplot(groups,labels=["Not Satisfied","Satisfied"],showfliers=False)
plt.title("Response Time by Satisfaction Group"); plt.xlabel("Satisfaction Group"); plt.ylabel("Response Time (minutes)")
plt.tight_layout(); plt.show()


A box plot shows the median, spread and long-tail of response time for satisfied versus non-satisfied interactions.


Item price is highly skewed, so price quintiles provide a stable comparison without allowing extreme values to dominate.


Satisfied interactions have a substantially lower typical response time than non-satisfied interactions. Long response-time tails indicate a service-level opportunity.


Average CSAT varies across price quintiles, but the operational effect is less clear than the response-time effect.


Reducing response delay is one of the most actionable findings. Faster response can reduce frustration, although issue complexity also matters.


Price segmentation may help identify high-value customer journeys, but price alone should not be treated as a cause of satisfaction.


#### Chart - 8

In [ ]:
plot_df=data.dropna(subset=["Item_price"]).copy()
plot_df["price_bin"]=pd.qcut(plot_df["Item_price"],q=5,duplicates="drop")
price_csat=plot_df.groupby("price_bin",observed=True)["CSAT Score"].mean()
plt.figure(figsize=(10,5)); plt.bar(range(len(price_csat)),price_csat.values)
plt.xticks(range(len(price_csat)),[str(x) for x in price_csat.index],rotation=25,ha="right")
plt.title("Average CSAT across Item-Price Quintiles"); plt.xlabel("Item Price Quintile"); plt.ylabel("Average CSAT"); plt.ylim(0,5)
plt.tight_layout(); plt.show()


Item price is highly skewed, so quintile grouping gives a stable comparison without letting extreme prices dominate the chart.


Remark availability is a simple way to compare interactions where customers provided written feedback versus those where they did not.


CSAT varies across price groups, but the difference is less operationally actionable than the response-time pattern.


Records with and without remarks show different average CSAT. This can reflect survey behavior or interaction type rather than an effect of the remark itself.


Price segmentation can help identify high-value customer journeys, but price alone should not be treated as a cause of satisfaction.


The insight can guide feedback-collection strategy, but should not be used to judge customers or agents based only on remark availability.


#### Chart - 9

In [ ]:
remark_stats=data.groupby("has_customer_remark")["CSAT Score"].mean()
plt.figure(figsize=(7,5)); plt.bar(["No Remark","Remark Available"],remark_stats.reindex([0,1]).values)
plt.title("Average CSAT by Customer Remark Availability"); plt.ylabel("Average CSAT"); plt.ylim(0,5)
plt.tight_layout(); plt.show()


Customer remarks are optional. Comparing remark availability helps understand whether feedback participation is associated with different CSAT levels.


Order-ID availability separates order-linked from non-order-linked interactions and provides a useful service-journey comparison.


Records with and without remarks show different average CSAT. This may reflect survey behavior or interaction type, so it is not evidence that remarks cause satisfaction.


Average CSAT differs between order-linked and non-order-linked records, likely reflecting different interaction types and issue mixes.


The result can inform feedback collection, but customers and agents should not be penalized based only on whether a remark exists.


This segmentation can inform service-journey design, but order presence should be considered together with issue category and channel.


#### Chart - 10

In [ ]:
order_stats=data.groupby("has_order")["CSAT Score"].mean()
plt.figure(figsize=(7,5)); plt.bar(["No Order ID","Order ID Available"],order_stats.reindex([0,1]).values)
plt.title("Average CSAT by Order Information Availability"); plt.ylabel("Average CSAT"); plt.ylim(0,5)
plt.tight_layout(); plt.show()


Order-ID availability is a simple proxy for whether the interaction is linked to an order record.


A line chart is appropriate for hourly observations because it makes time-of-day variation easy to see.


Average CSAT differs between order-linked and non-order-linked interactions, which may reflect differences in issue types and customer journeys.


Average CSAT changes across reporting hours, indicating that customer experience is not completely uniform during the day.


This segmentation can support service-journey design, but order presence should be interpreted together with issue category.


Staffing can be aligned with weaker periods, but hourly decisions should also consider interaction volume and response time.


#### Chart - 11

In [ ]:
hour_csat=data.groupby("issue_hour")["CSAT Score"].mean()
plt.figure(figsize=(11,5)); plt.plot(hour_csat.index,hour_csat.values,marker="o")
plt.title("Average CSAT by Issue Reporting Hour"); plt.xlabel("Hour of Day"); plt.ylabel("Average CSAT")
plt.xticks(range(24)); plt.ylim(0,5); plt.tight_layout(); plt.show()


A line chart is suitable for hourly data because it shows changes in satisfaction throughout the day.


A heatmap efficiently compares two categorical dimensions and highlights combinations with relatively low or high CSAT.


Average CSAT changes across reporting hours, indicating that service experience is not completely uniform throughout the day.


Channel performance varies by issue category. This indicates that channel strategy may need to be tailored to the type of customer problem.


Staffing capacity can be aligned with weaker periods, but hourly decisions should also consider interaction volume and response time.


Low-CSAT category-channel combinations are good candidates for specialized workflows, training and service-level review.


#### Chart - 12

In [ ]:
pivot=pd.pivot_table(data=data,index="category",columns="channel_name",values="CSAT Score",aggfunc="mean")
plt.figure(figsize=(10,7)); plt.imshow(pivot.values,aspect="auto",vmin=0,vmax=5,cmap="Blues")
plt.colorbar(label="Average CSAT"); plt.xticks(range(len(pivot.columns)),pivot.columns); plt.yticks(range(len(pivot.index)),pivot.index)
for r in range(pivot.shape[0]):
    for c in range(pivot.shape[1]):
        if pd.notna(pivot.iloc[r,c]): plt.text(c,r,f"{pivot.iloc[r,c]:.2f}",ha="center",va="center",fontsize=8)
plt.title("Average CSAT by Issue Category and Support Channel"); plt.xlabel("Channel"); plt.ylabel("Issue Category")
plt.tight_layout(); plt.show()


A heatmap efficiently compares two categorical dimensions and makes lower-performing category-channel combinations easy to identify.


Agent-level averages are useful for coaching, but a minimum survey threshold reduces instability caused by very small samples.


Channel performance varies by issue category, supporting the use of category-specific channel strategies.


There is meaningful variation in average CSAT across agents. High performers can provide best-practice examples and low performers can receive targeted coaching.


Low-CSAT category-channel combinations can be prioritized for specialized workflows, training, or service-level improvements.


Agent metrics should be interpreted with issue mix and workload context. They are most useful as a coaching signal rather than a standalone quality judgment.


#### Chart - 13

In [ ]:
agent_perf=data.groupby("Agent_name").agg(interactions=("CSAT Score","size"),avg_csat=("CSAT Score","mean"))
agent_perf=agent_perf[agent_perf["interactions"]>=30]
agent_plot=pd.concat([agent_perf.nsmallest(5,"avg_csat"),agent_perf.nlargest(5,"avg_csat")])
plt.figure(figsize=(11,6)); plt.barh(agent_plot.index,agent_plot["avg_csat"])
plt.title("Top and Bottom Agent CSAT (Minimum 30 Surveys)"); plt.xlabel("Average CSAT"); plt.ylabel("Agent"); plt.xlim(0,5)
plt.tight_layout(); plt.show()


Agent averages can be unstable for agents with few surveys, so the chart uses a minimum 30-survey threshold.


The heatmap summarizes relationships among numeric and engineered variables and helps detect redundancy before modeling.


There is meaningful variation in agent-level average CSAT. High performers can provide best-practice examples while low performers can receive targeted coaching.


Response time has the strongest practical relationship with satisfaction-related behavior among the operational numeric variables. Correlation does not establish causation.


Agent monitoring can support coaching and recognition, but should be adjusted for issue mix and should not be used as a standalone judgment of agent quality.


The correlation findings support feature selection and operational prioritization, while reminding us not to interpret pairwise correlation as a causal relationship.


#### Chart - 14 - Correlation Heatmap

In [ ]:
num_cols=["CSAT Score","response_mins_capped","item_price_log","remark_length","remark_word_count","has_customer_remark","has_order","handling_time_available","issue_hour","issue_dayofweek","is_weekend"]
corr=data[num_cols].corr()
plt.figure(figsize=(11,8)); plt.imshow(corr.values,cmap="coolwarm",vmin=-1,vmax=1,aspect="auto")
plt.colorbar(label="Correlation"); plt.xticks(range(len(num_cols)),num_cols,rotation=45,ha="right"); plt.yticks(range(len(num_cols)),num_cols)
plt.title("Correlation Heatmap"); plt.tight_layout(); plt.show()


A correlation heatmap summarizes linear relationships among numeric features and helps identify redundancy before modeling.


A correlation heatmap summarizes linear relationships among numeric features and supports identification of redundant predictors.


Response time shows a meaningful relationship with satisfaction-related variables, while several other engineered numeric features have weaker linear relationships. Correlation does not establish causation.


The numeric relationships are generally moderate; response time is the most operationally meaningful signal. Categorical effects are not captured fully by this heatmap.


#### Chart - 15 - Pair Plot

In [ ]:
pair_cols=["CSAT Score","response_mins_capped","item_price_log","remark_length","issue_hour"]
pair_data=data[pair_cols].dropna().sample(n=min(2000,len(data[pair_cols].dropna())),random_state=RANDOM_STATE)
pd.plotting.scatter_matrix(pair_data,figsize=(12,12),diagonal="hist",alpha=.25)
plt.suptitle("Pair Plot of Selected Numeric Features",y=.90); plt.show()


A pair plot helps inspect distributions and possible nonlinear or pairwise patterns among selected numeric variables.


A pair plot helps inspect distributions and pairwise relationships among a small set of numeric variables.


Response time is strongly skewed and several relationships are not purely linear, supporting the use of models that can capture nonlinear patterns.


Response time is highly skewed and several relationships are nonlinear or weak, supporting the use of tree-based models alongside a linear baseline.


## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

Three hypotheses are defined from the EDA:
1. Email and non-Email interactions have different mean CSAT.
2. Response-time distributions differ between satisfied and non-satisfied interactions.
3. CSAT distributions differ across agent shifts.
A 5% significance level is used.


### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

- **H0:** Mean CSAT for Email interactions equals mean CSAT for non-Email interactions.
- **H1:** Mean CSAT for Email interactions differs from mean CSAT for non-Email interactions.


#### 2. Perform an appropriate statistical test.

In [ ]:
email_scores=data.loc[data["channel_name"]=="Email","CSAT Score"]
non_email_scores=data.loc[data["channel_name"]!="Email","CSAT Score"]
t_stat,p_value=ttest_ind(email_scores,non_email_scores,equal_var=False)
print(f"Welch t-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.6e}")
print("Decision:", "Reject H0" if p_value<0.05 else "Fail to reject H0")


##### Which statistical test have you done to obtain P-Value?

Welch's independent two-sample t-test.


##### Why did you choose the specific statistical test?

The groups are independent and may have unequal variances and sizes. Welch's test is more robust than the equal-variance version for this comparison.


### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

- **H0:** Response-time distributions are the same for satisfied and non-satisfied interactions.
- **H1:** Response-time distributions differ between satisfied and non-satisfied interactions.


#### 2. Perform an appropriate statistical test.

In [ ]:
sat_response=data.loc[data["Satisfied"]==1,"response_mins"].dropna()
unsat_response=data.loc[data["Satisfied"]==0,"response_mins"].dropna()
u_stat,p_value=mannwhitneyu(sat_response,unsat_response,alternative="two-sided")
print(f"Mann-Whitney U statistic: {u_stat:.4f}")
print(f"P-value: {p_value:.6e}")
print(f"Median response time - satisfied: {sat_response.median():.2f} minutes")
print(f"Median response time - not satisfied: {unsat_response.median():.2f} minutes")
print("Decision:", "Reject H0" if p_value<0.05 else "Fail to reject H0")


##### Which statistical test have you done to obtain P-Value?

Mann-Whitney U test.


##### Why did you choose the specific statistical test?

Response time is strongly skewed with a long tail, so a non-parametric test is appropriate. The two satisfaction groups are independent.


### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

- **H0:** CSAT distributions are the same across all agent shifts.
- **H1:** At least one agent shift has a different CSAT distribution.


#### 2. Perform an appropriate statistical test.

In [ ]:
shift_groups=[g["CSAT Score"].values for _,g in data.groupby("Agent Shift")]
h_stat,p_value=kruskal(*shift_groups)
print(f"Kruskal-Wallis H statistic: {h_stat:.4f}")
print(f"P-value: {p_value:.6e}")
print("Decision:", "Reject H0" if p_value<0.05 else "Fail to reject H0")


##### Which statistical test have you done to obtain P-Value?

Kruskal-Wallis H test.


##### Why did you choose the specific statistical test?

There are five independent shift groups and CSAT is ordinal. Kruskal-Wallis avoids a normality assumption and tests whether at least one group differs.


## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation
print("Top missing-value fields:")
display(data.isna().sum().sort_values(ascending=False).head(10))

# Explicit representation for analysis-only text.
data["Customer Remarks_clean"]=data["Customer Remarks"].replace("","no_remark")
data["response_mins_capped"]=data["response_mins_capped"].fillna(data["response_mins_capped"].median())
data["item_price_log"]=data["item_price_log"].fillna(data["item_price_log"].median())


#### What all missing value imputation techniques have you used and why did you use those techniques?

Numerical variables use median imputation because several fields are skewed. Categorical variables use most-frequent imputation inside the model pipeline. Customer remarks use an explicit no-remark representation. Pipeline imputation is learned only from training data to prevent leakage.


### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments
def iqr_stats(s):
    s=s.dropna(); q1,q3=s.quantile([.25,.75]); iqr=q3-q1
    return {"Q1":q1,"Q3":q3,"IQR":iqr,"Lower":q1-1.5*iqr,"Upper":q3+1.5*iqr,
            "Outliers":int(((s<q1-1.5*iqr)|(s>q3+1.5*iqr)).sum())}
display(pd.DataFrame({"Item_price":iqr_stats(data["Item_price"]),
                      "Response_mins":iqr_stats(data["response_mins"])}).T)
print(f"99th-percentile response cap: {response_cap:.2f} minutes")
print(f"99th-percentile price cap: {price_cap:.2f}")


##### What all outlier treatment techniques have you used and why did you use those techniques?

Response time and item price are skewed, so extreme values are capped at the 99th percentile for modeling. Item price is log-transformed. Negative response times are treated as invalid data rather than valid outliers.


### 3. Categorical Encoding

In [ ]:
categorical_features=["channel_name","category","Sub-category","Product_category","Tenure Bucket","Agent Shift","Manager"]
numeric_features=["response_mins_capped","item_price_log","remark_length","remark_word_count",
                  "has_customer_remark","has_order","handling_time_available","issue_hour",
                  "issue_dayofweek","is_weekend"]
print("Categorical:",categorical_features)
print("Numeric:",numeric_features)


#### What all categorical encoding techniques have you used & why did you use those techniques?

One-hot encoding is used for nominal categorical variables. Tenure is kept categorical rather than forcing a linear relationship. High-cardinality Agent_name and Supervisor are excluded from the core model to reduce memorization and improve generalization.


### 4. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
# Expand Contraction
contractions={"can't":"cannot","won't":"will not","don't":"do not","didn't":"did not",
              "isn't":"is not","wasn't":"was not","couldn't":"could not","wouldn't":"would not",
              "i'm":"i am","it's":"it is","that's":"that is"}
def expand_contractions(text):
    text=text.lower()
    for short,expanded in contractions.items():
        text=re.sub(r"\b"+re.escape(short)+r"\b",expanded,text)
    return text
data["Customer Remarks_clean"]=data["Customer Remarks"].map(expand_contractions)
data[["Customer Remarks","Customer Remarks_clean"]].head()


#### 2. Lower Casing

In [ ]:
data["Customer Remarks_clean"]=data["Customer Remarks_clean"].str.lower()
data["Customer Remarks_clean"].head()


#### 3. Removing Punctuations

In [ ]:
data["Customer Remarks_clean"]=data["Customer Remarks_clean"].str.replace(r"[^a-zA-Z\s]"," ",regex=True)
data["Customer Remarks_clean"]=data["Customer Remarks_clean"].str.replace(r"\s+"," ",regex=True).str.strip()
data["Customer Remarks_clean"].head()


#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
data["Customer Remarks_clean"]=data["Customer Remarks_clean"].str.replace(r"https?://\S+|www\.\S+"," ",regex=True)
data["Customer Remarks_clean"]=data["Customer Remarks_clean"].apply(
    lambda s:" ".join(tok for tok in s.split() if not any(ch.isdigit() for ch in tok))
)
data["Customer Remarks_clean"]=data["Customer Remarks_clean"].str.replace(r"\s+"," ",regex=True).str.strip()
data["Customer Remarks_clean"].head()


#### 5. Removing Stopwords & Removing White spaces

In [ ]:
stop_words=set('a an and are as at be by for from has have he her i in is it its me my of on or our that the their this to was we were with you your'.split())
data["Customer Remarks_clean"]=data["Customer Remarks_clean"].apply(
    lambda s:" ".join(tok for tok in s.split() if tok not in stop_words)
)
data["Customer Remarks_clean"].head()


In [ ]:
data["Customer Remarks_clean"]=data["Customer Remarks_clean"].str.replace(r"\s+"," ",regex=True).str.strip()
data["Customer Remarks_clean"].head()


#### 6. Rephrase Text

In [ ]:
rephrase_map={"very good":"excellent","very bad":"poor","godd job":"good job",
               "issue not resolved":"problem unresolved","not resolved":"problem unresolved"}
def rephrase_text(text):
    for old,new in rephrase_map.items(): text=text.replace(old,new)
    return text
data["Customer Remarks_clean"]=data["Customer Remarks_clean"].map(rephrase_text)
data["Customer Remarks_clean"].head()


#### 7. Tokenization

In [ ]:
data["remark_tokens"]=data["Customer Remarks_clean"].apply(lambda s:s.split() if s else [])
data[["Customer Remarks_clean","remark_tokens"]].head()


#### 8. Text Normalization

In [ ]:
def simple_stem(token):
    for suffix in ("ing","ed","ly","es","s"):
        if len(token)>len(suffix)+3 and token.endswith(suffix):
            return token[:-len(suffix)]
    return token
data["normalized_tokens"]=data["remark_tokens"].apply(lambda toks:[simple_stem(t) for t in toks])
data["Customer Remarks_normalized"]=data["normalized_tokens"].apply(" ".join)
data[["Customer Remarks_clean","Customer Remarks_normalized"]].head()


##### Which text normalization technique have you used and why?

A lightweight suffix stemmer is used because it is deterministic and does not require downloading an external NLP corpus. It is sufficient for consistent text normalization in this structured-data project.


#### 9. Part of speech tagging

In [ ]:
def simple_pos(token):
    if token.endswith(("ing","ed")): return "VERB"
    if token.endswith("ly"): return "ADV"
    if token.endswith(("ous","ful","ive","al")): return "ADJ"
    return "OTHER"
data["pos_tags"]=data["remark_tokens"].apply(lambda toks:[(t,simple_pos(t)) for t in toks])
data[["remark_tokens","pos_tags"]].head()


#### 10. Text Vectorization

In [ ]:
text_data=data.loc[data["Customer Remarks_normalized"].str.len()>0,"Customer Remarks_normalized"]
tfidf=TfidfVectorizer(max_features=1000,ngram_range=(1,2),min_df=5,stop_words="english")
tfidf_matrix=tfidf.fit_transform(text_data)
print("TF-IDF matrix shape:",tfidf_matrix.shape)
print("Sample vocabulary:",list(tfidf.vocabulary_)[:20])


TF-IDF is used because it captures informative words and short phrases while reducing the weight of very common terms. The text representation is kept as an exploratory NLP component because customer remarks are missing for a large share of records.


TF-IDF is selected because it converts customer remarks into numerical features while emphasizing informative terms and down-weighting very common words. Unigrams and bigrams help capture short customer phrases.


### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Manipulate Features to minimize feature correlation and create new features
model_features=categorical_features+numeric_features
X=data[model_features].copy()
y=data["Satisfied"].copy()
corr=X[numeric_features].corr().abs()
upper=corr.where(np.triu(np.ones(corr.shape),k=1).astype(bool))
high_corr=[(a,b,round(upper.loc[b,a],2)) for a in upper.columns for b in upper.index
           if pd.notna(upper.loc[b,a]) and upper.loc[b,a]>=.80]
print("High-correlation numeric pairs (|r| >= .80):",high_corr)
print("Predictive feature count:",len(model_features))


#### 2. Feature Selection

In [ ]:
# Select features using business relevance, missingness, leakage risk and cardinality.
selected_features=model_features.copy()
print("Selected features:")
print(selected_features)


##### What all feature selection methods have you used  and why?

Feature selection uses business relevance, missingness, leakage risk, and cardinality. Unique id and Order_id are excluded because they identify records. Agent_name and Supervisor are excluded from the primary model because their high cardinality may encourage memorization rather than generalizable service patterns.


##### Which all features you found important and why?

Expected high-value features include response time, issue category/sub-category, support channel, product category, tenure, shift, and time-of-day. The final ranking is determined empirically by model explainability rather than by assumption.


### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

In [ ]:
print(data[["Item_price","item_price_log","response_mins","response_mins_capped"]].describe())
print("Item price uses log1p transformation; response time uses 99th-percentile capping.")


### 6. Data Scaling

In [ ]:
print("Numerical features:",numeric_features)
print("StandardScaler will be applied to numerical features for Logistic Regression only.")


##### Which method have you used to scale you data and why?

### 7. Dimesionality Reduction

Dimensionality reduction is not required for the final structured-feature model. After excluding high-cardinality identifiers, one-hot encoding creates a manageable feature space, and keeping the original features preserves business interpretability.


No PCA is applied. PCA would reduce interpretability, which is important when translating model results into operational actions.


In [ ]:
print("Dimensionality reduction skipped: structured feature space is manageable and interpretability is prioritized.")


No dimensionality-reduction technique is used for the final model. TF-IDF is demonstrated separately for text exploration.


No dimensionality-reduction technique is used for the final structured model. The one-hot feature space is manageable after excluding high-cardinality identifiers, and retaining original features improves business interpretability.


### 8. Data Splitting

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=.20,random_state=RANDOM_STATE,stratify=y
)
print("Train:",X_train.shape," Test:",X_test.shape)
print("Train class ratio:\n",y_train.value_counts(normalize=True).round(3))
print("Test class ratio:\n",y_test.value_counts(normalize=True).round(3))


##### What data splitting ratio have you used and why?

An 80:20 stratified split is used. It provides sufficient training data while retaining a large independent test set and preserves the minority/majority class ratio.


### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

Yes. Approximately 82.5% of interactions are Satisfied (CSAT 4-5) and 17.5% are Not Satisfied (CSAT 1-3). Accuracy alone would therefore be misleading.


In [ ]:
class_counts=y.value_counts().rename(index={0:"Not Satisfied",1:"Satisfied"})
class_pct=(y.value_counts(normalize=True)*100).round(2).rename(index={0:"Not Satisfied",1:"Satisfied"})
display(pd.DataFrame({"Count":class_counts,"Percentage":class_pct}))
print("Class weighting will be used for Logistic Regression and Random Forest.")


##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

Class weighting is preferred to naive oversampling because it preserves the original observations and avoids creating synthetic or duplicated records. It gives the minority class greater influence during training.


## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
# ML Model - 1: Logistic Regression
preprocessor_lr=ColumnTransformer([
    ("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),
                     ("onehot",OneHotEncoder(handle_unknown="ignore"))]),categorical_features),
    ("num",Pipeline([("imputer",SimpleImputer(strategy="median")),
                     ("scaler",StandardScaler())]),numeric_features)
])
lr_model=Pipeline([
    ("preprocessor",preprocessor_lr),
    ("classifier",LogisticRegression(max_iter=400,class_weight="balanced",solver="liblinear",random_state=RANDOM_STATE))
])
lr_model.fit(X_train,y_train)
lr_pred=lr_model.predict(X_test)
lr_prob=lr_model.predict_proba(X_test)[:,1]

def evaluate_model(name,y_true,pred,prob):
    return {"Model":name,"Accuracy":accuracy_score(y_true,pred),
            "Precision":precision_score(y_true,pred,zero_division=0),
            "Recall":recall_score(y_true,pred,zero_division=0),
            "F1":f1_score(y_true,pred,zero_division=0),
            "ROC_AUC":roc_auc_score(y_true,prob)}
lr_metrics=evaluate_model("Logistic Regression",y_test,lr_pred,lr_prob)
pd.DataFrame([lr_metrics]).round(4)


Logistic Regression provides an interpretable linear baseline. Class weighting improves attention to the minority Not Satisfied class. Its performance is a benchmark for the more flexible tree-based models.


In [ ]:
metric_names=["Accuracy","Precision","Recall","F1","ROC_AUC"]
plt.figure(figsize=(9,5))
plt.bar(metric_names,[lr_metrics[m] for m in metric_names])
plt.ylim(0,1); plt.title("Logistic Regression Evaluation Metrics"); plt.ylabel("Score")
plt.tight_layout(); plt.show()
print(classification_report(y_test,lr_pred,target_names=["Not Satisfied","Satisfied"]))


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Cross-validation & Hyperparameter Tuning — Logistic Regression
tune_size=min(15000,len(X_train))
tune_idx=train_test_split(X_train.index,train_size=tune_size,random_state=RANDOM_STATE,stratify=y_train)[0]
lr_grid=GridSearchCV(lr_model,{"classifier__C":[0.1,1.0]},scoring="f1",cv=2,n_jobs=1)
lr_grid.fit(X_train.loc[tune_idx],y_train.loc[tune_idx])
best_lr=lr_grid.best_estimator_
tuned_lr_pred=best_lr.predict(X_test); tuned_lr_prob=best_lr.predict_proba(X_test)[:,1]
tuned_lr_metrics=evaluate_model("Tuned Logistic Regression",y_test,tuned_lr_pred,tuned_lr_prob)
print("Best parameters:",lr_grid.best_params_)
pd.DataFrame([lr_metrics,tuned_lr_metrics]).round(4)


GridSearchCV with 2-fold cross-validation tunes regularization strength C. A small stratified subset is used to keep runtime practical while preserving both classes.


GridSearchCV with 2-fold cross-validation tunes Logistic Regression regularization strength (C). A representative stratified subset is used to keep the notebook efficient while retaining both target classes.


The tuned model is compared with the baseline on the untouched test set. F1 and ROC-AUC are emphasized because the target is imbalanced.


Tuning is retained when it improves or maintains F1/ROC-AUC without adding unnecessary complexity.


### ML Model - 2 — Random Forest

Random Forest is selected as a nonlinear ensemble model. It can capture interactions between channel, issue type, tenure, shift, and operational variables and does not require numerical scaling.


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# ML Model - 2 Implementation: Random Forest
preprocessor_rf=ColumnTransformer([
    ("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),
                     ("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]),categorical_features),
    ("num",Pipeline([("imputer",SimpleImputer(strategy="median"))]),numeric_features)
])
rf_model=Pipeline([
    ("preprocessor",preprocessor_rf),
    ("classifier",RandomForestClassifier(n_estimators=100,max_depth=14,min_samples_leaf=5,
                                         class_weight="balanced_subsample",random_state=RANDOM_STATE,n_jobs=-1))
])
rf_model.fit(X_train,y_train)
rf_pred=rf_model.predict(X_test)
rf_prob=rf_model.predict_proba(X_test)[:,1]
rf_metrics=evaluate_model("Random Forest",y_test,rf_pred,rf_prob)

metric_names=["Accuracy","Precision","Recall","F1","ROC_AUC"]
plt.figure(figsize=(9,5)); plt.bar(metric_names,[rf_metrics[m] for m in metric_names])
plt.ylim(0,1); plt.title("Random Forest Evaluation Metrics"); plt.ylabel("Score")
plt.tight_layout(); plt.show()
print(classification_report(y_test,rf_pred,target_names=["Not Satisfied","Satisfied"]))
pd.DataFrame([rf_metrics]).round(4)


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Cross-validation & Hyperparameter Tuning — Random Forest
rf_grid=GridSearchCV(
    rf_model,
    {"classifier__max_depth":[10,14],"classifier__min_samples_leaf":[3,5]},
    scoring="f1",cv=2,n_jobs=1
)
tune_size=min(15000,len(X_train))
tune_idx=train_test_split(X_train.index,train_size=tune_size,random_state=RANDOM_STATE,stratify=y_train)[0]
rf_grid.fit(X_train.loc[tune_idx],y_train.loc[tune_idx])
best_rf=rf_grid.best_estimator_
tuned_rf_pred=best_rf.predict(X_test); tuned_rf_prob=best_rf.predict_proba(X_test)[:,1]
tuned_rf_metrics=evaluate_model("Tuned Random Forest",y_test,tuned_rf_pred,tuned_rf_prob)
print("Best parameters:",rf_grid.best_params_)
pd.DataFrame([rf_metrics,tuned_rf_metrics]).round(4)


GridSearchCV tests tree depth and minimum leaf size because these directly control Random Forest complexity and overfitting.


GridSearchCV tests tree depth and minimum leaf size because they control Random Forest complexity and overfitting. The selected configuration is evaluated on the same untouched test set.


The tuned Random Forest is evaluated on the same untouched test set. F1 is the tuning objective, with ROC-AUC reported as a secondary measure.


The tuned model is retained when it improves or maintains minority-class F1 and overall ranking quality.


#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

Accuracy shows overall correctness but is inflated by the majority class. Precision measures the proportion of flagged interactions that are actually Not Satisfied. Recall measures how many truly Not Satisfied interactions are identified and is important for service recovery. F1 balances precision and recall. ROC-AUC measures ranking quality across thresholds. For this business problem, recall and F1 are the most operationally relevant metrics.


### ML Model - 3

In [ ]:
# ML Model - 3: HistGradientBoosting
preprocessor_hgb=ColumnTransformer([
    ("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),
                     ("ordinal",OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1))]),categorical_features),
    ("num",Pipeline([("imputer",SimpleImputer(strategy="median"))]),numeric_features)
])
hgb_model=Pipeline([
    ("preprocessor",preprocessor_hgb),
    ("classifier",HistGradientBoostingClassifier(max_iter=120,learning_rate=.08,max_leaf_nodes=31,
                                                   l2_regularization=1.0,class_weight="balanced",
                                                   random_state=RANDOM_STATE))
])
hgb_model.fit(X_train,y_train)
hgb_pred=hgb_model.predict(X_test); hgb_prob=hgb_model.predict_proba(X_test)[:,1]
hgb_metrics=evaluate_model("HistGradientBoosting",y_test,hgb_pred,hgb_prob)
pd.DataFrame([hgb_metrics]).round(4)


HistGradientBoosting is a gradient-boosted tree model that can capture nonlinear patterns while remaining efficient on a compact structured feature set.


In [ ]:
metric_names=["Accuracy","Precision","Recall","F1","ROC_AUC"]
plt.figure(figsize=(9,5))
plt.bar(metric_names,[hgb_metrics[m] for m in metric_names])
plt.ylim(0,1); plt.title("HistGradientBoosting Evaluation Metrics"); plt.ylabel("Score")
plt.tight_layout(); plt.show()
print(classification_report(y_test,hgb_pred,target_names=["Not Satisfied","Satisfied"]))


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Cross-validation & Hyperparameter Tuning — HistGradientBoosting
hgb_grid=GridSearchCV(
    hgb_model,
    {"classifier__learning_rate":[.05,.08],
     "classifier__max_leaf_nodes":[15,31]},
    scoring="f1",cv=2,n_jobs=1
)
tune_size=min(15000,len(X_train))
tune_idx=train_test_split(X_train.index,train_size=tune_size,random_state=RANDOM_STATE,stratify=y_train)[0]
hgb_grid.fit(X_train.loc[tune_idx],y_train.loc[tune_idx])
best_hgb=hgb_grid.best_estimator_
tuned_hgb_pred=best_hgb.predict(X_test); tuned_hgb_prob=best_hgb.predict_proba(X_test)[:,1]
tuned_hgb_metrics=evaluate_model("Tuned HistGradientBoosting",y_test,tuned_hgb_pred,tuned_hgb_prob)
print("Best parameters:",hgb_grid.best_params_)
pd.DataFrame([hgb_metrics,tuned_hgb_metrics]).round(4)


GridSearchCV tests learning rate and tree complexity. These parameters balance model flexibility, learning speed, and overfitting.


The tuned model is evaluated on the untouched test set using F1 and ROC-AUC, with recall retained as an important business metric.


The tuned configuration is preferred when it improves or maintains minority-class F1/ROC-AUC without unnecessary complexity.


GridSearchCV tests learning rate and maximum leaf nodes for HistGradientBoosting. The tuned model is retained when it improves or maintains F1 and ROC-AUC.


The primary metrics are **Recall, F1-score, and ROC-AUC**, with accuracy and precision as supporting metrics. Recall is important because missing an unhappy customer can allow dissatisfaction to continue. F1 balances recall with precision, while ROC-AUC measures ranking quality across thresholds.


Recall, F1-score and ROC-AUC are prioritized because the target is imbalanced. Recall is important for identifying as many dissatisfied interactions as possible, while F1 balances recall and precision. Accuracy and precision are supporting metrics.


The final prediction model is chosen using the highest test-set F1, with ROC-AUC as the secondary criterion. This is more appropriate than selecting solely on accuracy because the target is imbalanced. The selected pipeline is then retained for deployment-oriented sanity checking.


The final model is selected using the highest F1-score, with ROC-AUC as a secondary criterion. This is appropriate because accuracy can look strong on an imbalanced target even when the minority class is poorly identified. The chosen model provides the best balance between finding unsatisfied customers and avoiding excessive false alerts.


Feature importance is extracted from the selected Random Forest when a tree-based model wins, using its one-hot encoded feature names. For a non-Random-Forest winner, permutation importance is calculated on a representative test sample. This connects the model to operational drivers such as response time, issue category, channel, tenure, product category, and shift.


In [ ]:
# Compare models and explain the selected model
results=pd.DataFrame([lr_metrics,tuned_lr_metrics,rf_metrics,tuned_rf_metrics,hgb_metrics,tuned_hgb_metrics])
results=results.drop_duplicates("Model").sort_values(["F1","ROC_AUC"],ascending=False).reset_index(drop=True)
display(results.round(4))

model_lookup={
    "Logistic Regression":lr_model,
    "Tuned Logistic Regression":best_lr,
    "Random Forest":rf_model,
    "Tuned Random Forest":best_rf,
    "HistGradientBoosting":hgb_model,
    "Tuned HistGradientBoosting":best_hgb
}
best_name=results.loc[0,"Model"]
final_model=model_lookup[best_name]
print("Selected final model:",best_name)

# Explainability
if "Random Forest" in best_name:
    fitted_pre=final_model.named_steps["preprocessor"]
    fitted_clf=final_model.named_steps["classifier"]
    names=fitted_pre.get_feature_names_out()
    importances=pd.Series(fitted_clf.feature_importances_,index=names).sort_values(ascending=False).head(20)
    display(importances.to_frame("Importance"))
    plt.figure(figsize=(10,7)); plt.barh(importances.index,importances.values)
    plt.title(f"Top Feature Importances — {best_name}"); plt.xlabel("Importance"); plt.ylabel("Feature")
    plt.tight_layout(); plt.show()
else:
    sample=X_test.sample(n=min(1200,len(X_test)),random_state=RANDOM_STATE)
    sample_y=y_test.loc[sample.index]
    perm=permutation_importance(final_model,sample,sample_y,n_repeats=2,random_state=RANDOM_STATE,
                                scoring="f1",n_jobs=1)
    perm_imp=pd.Series(perm.importances_mean,index=X_test.columns).sort_values(ascending=False).head(15)
    display(perm_imp.to_frame("Permutation Importance"))


## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
# Save the File
model_output=Path("final_csat_model.joblib")
joblib.dump(final_model,model_output)
print(f"Saved final model to: {model_output.resolve()}")


### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Load the File and predict unseen data for a sanity check
loaded_model=joblib.load(model_output)
unseen=X_test.head(5).copy()
pred=loaded_model.predict(unseen)
prob=loaded_model.predict_proba(unseen)[:,1]
sanity=unseen.copy()
sanity["Actual_Satisfied"]=y_test.head(5).values
sanity["Predicted_Satisfied"]=pred
sanity["Predicted_Probability"]=prob.round(4)
sanity


### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

## Conclusion

The analysis covers 85,907 customer-support interactions and shows that the CSAT distribution is heavily concentrated at score 5. For actionable classification, CSAT 4-5 is treated as Satisfied and CSAT 1-3 as Not Satisfied. This produces a meaningful minority class and makes recall and F1 more informative than accuracy alone.

EDA identifies differences across support channels, issue categories, product categories, agent tenure, shifts, and operational response time. Email has a lower average CSAT than Inbound and Outcall. Satisfied interactions also have a substantially lower typical response time than non-satisfied interactions. These observations support focusing on response SLAs, channel-specific service design, and targeted improvements for lower-performing issue types.

The data-preparation workflow addresses duplicates, timestamp parsing, invalid response times, missing values, outliers, feature engineering, categorical encoding, numerical scaling, text preprocessing, and class imbalance. Customer remarks are cleaned and represented using TF-IDF for exploratory NLP, while the main predictive model uses structured features because remarks and several other operational fields are highly incomplete.

Logistic Regression, Random Forest, and HistGradientBoosting are compared and tuned using cross-validation. The final model is selected based on F1-score with ROC-AUC as a secondary criterion. Model explainability identifies the features that contribute most to predictions, allowing the analytical output to be translated into operational actions.

Recommended actions include reducing response delays, investigating low-performing channel/category combinations, strengthening onboarding and coaching, sharing best practices from high-performing teams, and using the model as a proactive CSAT-risk signal. The model should support—not replace—human operational decisions and should be monitored for drift as customer behavior and service processes change. Future work can add richer resolution/outcome fields and a dedicated sentiment model for customer remarks.


### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***